In [1]:
import pandas as pd
import numpy as np
from CCA_utils import *

## Baseline Model Panel

In [23]:
study_sovereigns = [
    'Saudi Arabia', 'UAE (Abu Dhabi)', 'Qatar', 'Colombia',
    'Mexico', 'Brazil', 'Egypt', 'Malaysia','Indonesia', 'Philippines', 'Turkey', 'Chile', 'China',
    'South Africa', 'South Korea', 'Thailand']

cca_panel_df = pd.read_csv('../data/processed/CCA_V2/CCA_panel.csv')
cca_panel_df['date'] = pd.to_datetime(cca_panel_df['date'])
cca_panel_df = cca_panel_df[cca_panel_df['country'].isin(study_sovereigns)].copy()

T = 5.0
vol_window = 52
freq = 'W'

cca_panel_df.set_index(['date','country'], inplace=True)
cca_panel_df = (
    cca_panel_df
    .groupby('country')
    .resample(freq, level='date')
    .last()
)
cca_panel_df.reset_index(inplace=True)


## M2 Specific Data

In [24]:
ovx_df = pd.read_csv('../data/processed/Macroeconomic_variables/OVXCLS.csv')
ovx_df['date'] = pd.to_datetime(ovx_df['date'])
ovx_df = ovx_df.sort_values('date')
ovx_df['OVXCLS'] = ovx_df['OVXCLS'].ffill()

cca_panel_df = cca_panel_df.reset_index()
cca_panel_df = cca_panel_df.sort_values('date')

cca_panel_df = pd.merge_asof(
    cca_panel_df,
    ovx_df[['date', 'OVXCLS']],
    on='date',
    direction='backward'
)

cca_panel_df.set_index(['date', 'country'], inplace=True)
cca_panel_df = (
    cca_panel_df
    .groupby('country')
    .resample(freq, level='date')
    .last()
)
cca_panel_df.reset_index(inplace=True)
print(f"OVX NAs: {cca_panel_df['OVXCLS'].isna().sum()}")

OVX NAs: 0


## M2 Parameters & Pricer

In [25]:
# --- Sigmoid: OVX -> annualized jump intensity ---
# lambda(OVX) = L / (1 + exp(-k * (OVX - x0)))
logit_const = -5.1623
logit_beta  = 0.0409
lambda_min_annual = 0

def ovx_to_lambda(ovx):
    if np.isnan(ovx):
        return 0.0
    logit_p = logit_const + logit_beta * ovx
    lambda_weekly = 1 / (1 + np.exp(-logit_p))
    lambda_annual = lambda_weekly * 52
    return lambda_annual

# --- Fixed jump size ---
gamma = 0.05
J = -0.1366

# --- Initialize pricer ---
pricer = FixedJumpCCAPricer(J=J*gamma, max_terms=20)

## Run M2

In [26]:
results = pd.DataFrame()

print("Starting M2 calibration...")

for country, group in cca_panel_df.groupby('country'):
    print(f"Processing {country}...")
    df = group.copy().sort_values('date').reset_index(drop=True)

    r_d = df['domestic_rate']
    r_f = df['risk_free_rate']
    M_bn = df['monetary_base_bn_local']
    dom_D_bn = df['domestic_debt_bn_local']
    ext_D_bn = df['external_debt_bn_usd']
    fx_rate = df['fx_rate']

    df['LCL_usd'] = [
        compute_lcl_usd(m, bd, fx, rd, rf, T)
        for m, bd, fx, rd, rf in zip(M_bn, dom_D_bn, fx_rate, r_d, r_f)
    ]

    ann_factor = np.sqrt(52) if freq == 'W' else np.sqrt(12)
    log_ret = np.log(df['LCL_usd'] / df['LCL_usd'].shift(1))
    df['sigma_lcl'] = log_ret.rolling(window=vol_window).std() * ann_factor

    df['B_f'] = [
        compute_barrier_kvm(debt, rf, T)
        for debt, rf in zip(ext_D_bn, r_f)
    ]
    
    df['lambda_annual'] = df['OVXCLS'].apply(ovx_to_lambda)

    out = {'implied_V': [], 'implied_sigma_V': [], 'cca_converged': []}

    for i, row in df.iterrows():
        lam = row['lambda_annual']

        cca_base = solve_CCA(row['LCL_usd'], row['sigma_lcl'], row['B_f'], r_f.iloc[i], T)
        v_guess   = cca_base['V']       if cca_base['converged'] else (row['LCL_usd'] + row['B_f'])
        sig_guess = cca_base['sigma_V'] if cca_base['converged'] else (row['sigma_lcl'] * row['LCL_usd'] / (row['LCL_usd'] + row['B_f']))

        cca = pricer.solve_CCA(row['LCL_usd'], row['sigma_lcl'], row['B_f'],
                               r_f.iloc[i], T, lam,
                               v_guess=v_guess, sig_guess=sig_guess)

        out['implied_V'].append(cca['V'])
        out['implied_sigma_V'].append(cca.get('sigma_total', cca.get('sigma_diff', np.nan)))
        out['cca_converged'].append(cca['converged'])

    for col, vals in out.items():
        df[col] = vals

    results = pd.concat([results, df])

START_DATE = '2015-01-01'
END_DATE = '2024-12-31'
results = results[
    (results['date'] >= START_DATE) & (results['date'] <= END_DATE)
].copy()

print("Calibration complete!")

Starting M2 calibration...
Processing Brazil...
Processing Chile...
Processing China...
Processing Colombia...
Processing Egypt...
Processing Indonesia...
Processing Malaysia...
Processing Mexico...
Processing Philippines...
Processing Qatar...
Processing Saudi Arabia...
Processing South Africa...
Processing South Korea...
Processing Thailand...
Processing Turkey...
Processing UAE (Abu Dhabi)...
Calibration complete!


In [27]:
failed = results[~results['cca_converged']][['country', 'date', 'LCL_usd', 'B_f', 'sigma_lcl', 'risk_free_rate']].copy()

for _, row in failed.iterrows():
    m0 = solve_CCA(row['LCL_usd'], row['sigma_lcl'], row['B_f'], row['risk_free_rate'], T)
    print(f"{row['country']:<20} {str(row['date'].date())} conv={m0['converged']} V={m0.get('V', float('nan')):.2f} sig={m0.get('sigma_V', float('nan')):.4f}")

In [28]:
results[['date', 'country', 'cds_spread', 'risk_free_rate',
         'implied_V', 'implied_sigma_V', 'cca_converged',
         'B_f', 'LCL_usd', 'sigma_lcl',
         'lambda_annual']].to_csv(
    '../output/results/M2_results_5YCDS_weekly_05_dampening.csv', index=False)

In [56]:
gamma = 1
J = -0.136

# --- Initialize pricer ---
pricer = FixedJumpCCAPricer(J=J*gamma, max_terms=20)

In [62]:
debt_to_equity = 1.64
LCL_usd = 100 

sigma_lcl = 0.1
B_f = 700
r_f = 0.05
T = 5
OVX = 50
lam = ovx_to_lambda(OVX)
print(lam)
cca = pricer.solve_CCA(LCL_usd, sigma_lcl, B_f, r_f, T, lam, v_guess=LCL_usd + B_f, sig_guess=sigma_lcl)

print(cca)

DD = (np.log(cca['V'] / B_f) + (r_f - 0.5* cca['sigma_total']**2) * T)/cca['sigma_total'] * np.sqrt(T)

print(f"Distance to Default: {DD:.4f}")


2.2047615574821893
{'V': np.float64(507.4780808574664), 'sigma_diff': np.float64(1.0000000010422818e-06), 'sigma_total': np.float64(0.2170582159163066), 'converged': True}
Distance to Default: -1.9513
